# Processamento — carteira_auto v0.2.1

Este notebook documenta os componentes de processamento:
**Result type → Pydantic Models → Analyzers → Alerts**

Cobre:
1. Result type (Ok/Err) — error handling explicito
2. Pydantic Models — validacao estrita
3. Os 10 Analyzers — metricas de dominio
4. Sistema de Alertas — regras e canais
5. Error tracking parcial — ctx["_errors"]

## 1. Result Type — Error Handling Explicito

`Ok[T]` e `Err[T]` permitem propagar erros sem excecoes.
Inspirado no Result do Rust.

In [ ]:
from carteira_auto.core import Ok, Err

# Sucesso
resultado = Ok(42)
print(f"is_ok: {resultado.is_ok()}")
print(f"unwrap: {resultado.unwrap()}")

# Erro
erro = Err("Dados insuficientes", details={"min_required": 30, "actual": 5})
print(f"\nis_ok: {erro.is_ok()}")
print(f"error: {erro.error}")
print(f"details: {erro.details}")
print(f"unwrap_or(0): {erro.unwrap_or(0)}")

In [ ]:
# Uso pratico: funcao que pode falhar
from carteira_auto.core.result import Result

def calcular_retorno(atual: float, investido: float) -> Result[float]:
    """Calcula retorno percentual. Retorna Err se investido == 0."""
    if investido == 0:
        return Err("Valor investido nao pode ser zero")
    return Ok((atual - investido) / investido * 100)

# Caso de sucesso
r1 = calcular_retorno(1200, 1000)
print(f"Retorno: {r1.unwrap():.1f}%")

# Caso de erro
r2 = calcular_retorno(1200, 0)
print(f"Erro: {r2.error}")
print(f"Fallback: {r2.unwrap_or(0.0):.1f}%")

## 2. Pydantic Models — Validacao Estrita

Todos os modelos de dominio usam Pydantic com validacao via `field_validator`.
Campos opcionais usam `float | None = None` (graceful degradation).

In [ ]:
from carteira_auto.core.models import Asset, Portfolio

# Criar ativo valido
ativo = Asset(
    ticker="PETR4",
    nome="Petrobras PN",
    preco_atual=35.50,
    preco_medio=28.00,
    posicao_atual=5000.0,
    n_cotas_atual=142,
)
print(f"Ativo: {ativo.ticker} — R${ativo.preco_atual}")

In [ ]:
from pydantic import ValidationError

# Validacao: ticker vazio -> erro
try:
    Asset(ticker="", nome="Invalido")
except ValidationError as e:
    print(f"Ticker vazio: {e.errors()[0]['msg']}")

# Validacao: preco negativo -> erro
try:
    Asset(ticker="VALE3", nome="Vale", preco_atual=-10.0)
except ValidationError as e:
    print(f"Preco negativo: {e.errors()[0]['msg']}")

In [ ]:
# Imutabilidade via model_copy() — nunca mutar in-place
ativo_atualizado = ativo.model_copy(update={"preco_atual": 38.00})
print(f"Original: R${ativo.preco_atual}")
print(f"Copia atualizada: R${ativo_atualizado.preco_atual}")

In [ ]:
# Portfolio com multiplos ativos
portfolio = Portfolio(assets=[
    Asset(ticker="PETR4", nome="Petrobras PN", posicao_atual=5000, preco_atual=35.50, preco_medio=28.0, pct_meta=0.15, n_cotas_atual=142),
    Asset(ticker="VALE3", nome="Vale ON", posicao_atual=3000, preco_atual=62.00, preco_medio=55.0, pct_meta=0.10, n_cotas_atual=48),
    Asset(ticker="ITUB4", nome="Itau PN", posicao_atual=4000, preco_atual=32.00, preco_medio=30.0, pct_meta=0.12, n_cotas_atual=125),
])
print(f"Portfolio com {len(portfolio.assets)} ativos")
for a in portfolio.assets:
    print(f"  {a.ticker}: R${a.posicao_atual:,.0f} (meta: {a.pct_meta:.0%})")

### Models de Metricas

Cada analyzer produz um model Pydantic especifico:

| Model | Produzido por | Campos principais |
|-------|---------------|-------------------|
| PortfolioMetrics | PortfolioAnalyzer | total_value, total_return_pct, allocations |
| RiskMetrics | RiskAnalyzer | var_95, sharpe_ratio, max_drawdown, beta |
| MacroContext | MacroAnalyzer | selic, cdi, ipca, pib_growth (12 campos) |
| MarketMetrics | MarketAnalyzer | ibov, ifix, cdi_acum (8 campos) |
| CurrencyMetrics | CurrencyAnalyzer | usd_brl, dxy, carry_spread (11 campos) |
| CommodityMetrics | CommodityAnalyzer | oil_brent, gold, soybean, cycle_signal (18 campos) |
| FiscalMetrics | FiscalAnalyzer | divida_bruta_pib, fiscal_trajectory (9 campos) |

In [ ]:
from carteira_auto.core.models import (
    CurrencyMetrics, CommodityMetrics, FiscalMetrics,
    MacroContext, RiskMetrics,
)

# Todos os campos sao Optional — graceful degradation
fiscal = FiscalMetrics(
    divida_bruta_pib=78.5,
    resultado_primario_pib=0.43,
    fiscal_trajectory="warning",
    summary="Divida bruta 78.5% PIB; primario +0.43% PIB; trajetoria=warning",
)
print(f"Fiscal: {fiscal.summary}")
print(f"Divida liquida (nao informada): {fiscal.divida_liquida_pib}")

## 3. Analyzers — 10 Nodes de Analise

Cada analyzer e um `Node` do DAG que le dados do `PipelineContext`,
calcula metricas e escreve modelos Pydantic de volta.

| Analyzer | Node name | Deps | ctx key |
|----------|-----------|------|---------|
| PortfolioAnalyzer | analyze_portfolio | [fetch_portfolio_prices] | portfolio_metrics |
| RiskAnalyzer | analyze_risk | [fetch_portfolio_prices, analyze_portfolio] | risk_metrics |
| MacroAnalyzer | analyze_macro | [] | macro_context |
| MarketAnalyzer | analyze_market | [] | market_metrics |
| MarketSectorAnalyzer | analyze_market_sectors | [] | market_sectors |
| EconomicSectorAnalyzer | analyze_economic_sectors | [] | economic_sectors |
| Rebalancer | rebalance | [analyze_portfolio] | rebalance_recommendations |
| CurrencyAnalyzer | analyze_currency | [] | currency_metrics |
| CommodityAnalyzer | analyze_commodities | [] | commodity_metrics |
| FiscalAnalyzer | analyze_fiscal | [] | fiscal_metrics |

In [ ]:
from carteira_auto.analyzers import (
    PortfolioAnalyzer, RiskAnalyzer, MacroAnalyzer,
    MarketAnalyzer, Rebalancer, MarketSectorAnalyzer,
    EconomicSectorAnalyzer, CurrencyAnalyzer,
    CommodityAnalyzer, FiscalAnalyzer,
)

# Todos os analyzers registrados
analyzers = [
    PortfolioAnalyzer, RiskAnalyzer, MacroAnalyzer,
    MarketAnalyzer, Rebalancer, MarketSectorAnalyzer,
    EconomicSectorAnalyzer, CurrencyAnalyzer,
    CommodityAnalyzer, FiscalAnalyzer,
]

print(f"{len(analyzers)} analyzers disponiveis:\n")
for cls in analyzers:
    node = cls()
    deps = node.dependencies if node.dependencies else "(nenhuma)"
    print(f"  {node.name:30s} deps={deps}")

In [ ]:
# Executar um analyzer sem dependencias via PipelineContext
# Nota: analyzers com deps=[] buscam dados diretamente dos fetchers
# (fazem lazy import dentro do run())

from carteira_auto.core.engine import PipelineContext

# FiscalAnalyzer nao tem dependencias — busca do BCB diretamente
# CUIDADO: faz chamadas reais a API do BCB
fiscal_analyzer = FiscalAnalyzer()
ctx = PipelineContext()
ctx = fiscal_analyzer.run(ctx)

metrics = ctx["fiscal_metrics"]
print(f"Divida bruta/PIB: {metrics.divida_bruta_pib}%")
print(f"Resultado primario: {metrics.resultado_primario_pib}%")
print(f"Trajetoria fiscal: {metrics.fiscal_trajectory}")
print(f"Sumario: {metrics.summary}")

## 4. Sistema de Alertas

O AlertEngine avalia regras contra o PipelineContext e gera alertas.
Cada alerta tem severidade e pode ser enviado por diferentes canais.

In [ ]:
from carteira_auto.alerts.engine import AlertEngine, AlertRule
from carteira_auto.alerts.channels import ConsoleChannel

# Criar engine de alertas
alert_engine = AlertEngine()

# Registrar regra: alerta se desvio de alocacao > 5%
alert_engine.register_rule(AlertRule(
    name="desvio_alocacao",
    condition="deviation_above",
    threshold=0.05,
    severity="warning",
    message_template="Classe {asset_class} desviou {deviation:.1%} da meta",
))

print(f"Regras registradas: {len(alert_engine.rules)}")
for rule in alert_engine.rules:
    print(f"  {rule.name}: {rule.condition} > {rule.threshold} ({rule.severity})")

## 5. Error Tracking Parcial

Analyzers usam error tracking parcial: cada indicador pode falhar
independentemente sem derrubar o node inteiro.

Erros ficam em `ctx["_errors"]["node_name.partial"]`.

In [ ]:
# Padrao de error tracking (usado por todos os analyzers)
from carteira_auto.core.engine import Node, PipelineContext
import traceback

class ExemploAnalyzer(Node):
    """Demonstra error tracking parcial."""
    name = "exemplo"
    dependencies: list[str] = []

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.setdefault("_errors", {})
        errors: list[str] = []

        # Indicador A — sucesso
        valor_a = None
        try:
            valor_a = 42.0
        except Exception as e:
            errors.append(f"A: {e}")

        # Indicador B — falha
        valor_b = None
        try:
            raise ConnectionError("API indisponivel")
        except Exception as e:
            errors.append(f"B: {e}")

        if errors:
            ctx["_errors"]["exemplo.partial"] = "; ".join(errors)

        ctx["exemplo_metrics"] = {"a": valor_a, "b": valor_b}
        return ctx

ctx = PipelineContext()
ctx = ExemploAnalyzer().run(ctx)

print(f"Metricas: {ctx['exemplo_metrics']}")
print(f"Erros parciais: {ctx['_errors']}")
print(f"\nNota: valor_a=42.0 (sucesso), valor_b=None (falhou) — pipeline nao interrompido")

## Resumo

O fluxo de processamento do carteira_auto:

```
Dados brutos (fetchers/DataLake)
       |
       v
Pydantic Models — validacao na entrada (Asset, Portfolio)
       |
       v
10 Analyzers — cada um produz um *Metrics model no ctx
       |
       v
Error tracking — erros parciais em ctx['_errors']
       |
       v
AlertEngine — avalia regras, gera alertas
       |
       v
Result type — Ok/Err para error handling explicito
```

Proximos notebooks (examples/):
- `01_macro_brasil.ipynb` — analise macro com dados reais
- `02_macro_global.ipynb` — Fed, Treasuries, VIX, DXY
- `03_renda_fixa.ipynb` — Tesouro Direto
- `04_acoes_fundamentos.ipynb` — acoes e fundamentos
- `05_portfolio_pipeline.ipynb` — pipeline completo